
# Vertex AI Project 1 — Hello Vertex: Deploy a Tiny Model and Predict

This notebook walks through:
1) Training a tiny scikit-learn model locally  
2) Uploading the model to **Vertex AI Model Registry**  
3) Deploying the model to a **Vertex AI Endpoint** (online predictions)  
4) Sending **online** and **batch** predictions  
5) (TBD) Enabling basic **logging**  
6) (Optional) Cleanup resources (undeploy endpoint, delete endpoint)



## 0) Prerequisites



In [1]:
!pip install --upgrade pip
!pip install packaging setuptools wheel
# Pin scikit-learn to a version compatible with the serving container
!pip -q install -U google-cloud-aiplatform "scikit-learn~=1.4.0" joblib google-cloud-storage

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 25.6.0 requires scikit-learn>=1.5, but you have scikit-learn 1.4.2 which is incompatible.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.


In [2]:
!pip show scikit-learn google-cloud-aiplatform joblib

Name: scikit-learn
Version: 1.4.2
Summary: A set of python modules for machine learning and data mining
Home-page: https://scikit-learn.org
Author: 
Author-email: 
License: new BSD
Location: /usr/local/lib/python3.11/dist-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: cuml-cu12, fastai, hdbscan, imbalanced-learn, libpysal, librosa, mlxtend, pynndescent, sentence-transformers, shap, sklearn-compat, sklearn-pandas, tsfresh, umap-learn, yellowbrick
---
Name: google-cloud-aiplatform
Version: 1.120.0
Summary: Vertex AI API client library
Home-page: https://github.com/googleapis/python-aiplatform
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /usr/local/lib/python3.11/dist-packages
Requires: docstring_parser, google-api-core, google-auth, google-cloud-bigquery, google-cloud-resource-manager, google-cloud-storage, google-genai, packaging, proto-plus, protobuf, pydantic, shapely, typing_extensions
Required-by: 
---
Name: jobli


## 1) Configure the environment


In [12]:

from google.cloud import aiplatform
from google.cloud import storage
import joblib, os, json, time, uuid, pathlib, pickle, sklearn
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from google.cloud import aiplatform

PROJECT_ID = "instr-cs795-fall25-hqin-1"
LOCATION   = "us-east4"
BUCKET     = "gs://instr-cs795-fall25-hqin-1-arasm002"

# Derived/utility
TIMESTAMP = time.strftime("%Y%m%d-%H%M%S")
DISPLAY_NAME = f"iris-rf-sklearn-{TIMESTAMP}"
MODEL_DIR  = f"model-artifacts-{TIMESTAMP}"
MODEL_FILE = f"{MODEL_DIR}/model.pkl"
BATCH_INPUT_DIR = f"batch_inputs_{TIMESTAMP}"
GCS_INPUT_PREFIX = f"{BUCKET}/batch_inputs/{TIMESTAMP}/"
GCS_OUTPUT_PREFIX = f"{BUCKET}/batch_outputs/{TIMESTAMP}/"

print("Project:", PROJECT_ID)
print("Location:", LOCATION)
print("Bucket:", BUCKET)


Project: instr-cs795-fall25-hqin-1
Location: us-east4
Bucket: gs://instr-cs795-fall25-hqin-1-arasm002


### 1.1) Verify scikit-learn version

Let's confirm the version of scikit-learn loaded in the environment before training. It should be `1.4.x` to match the serving container.


## TODO: Enable logging & view logs  ✅

> Note: I put in a ticket for access to logging because I was getting a permissions error

Vertex AI sends container logs to **Cloud Logging**. You can also enable **access logging** from the console.
- In Cloud Console, go to **Logging → Logs Explorer** and filter on your endpoint or model resource name.
- For access logging (request/response metadata), enable it on the Endpoint's **Monitoring** tab.


In [ ]:
import sklearn
print(f"Using scikit-learn version: {sklearn.__version__}")

Using scikit-learn version: 1.4.2



## 2) Train a tiny model locally
We'll train a small RandomForest on the Iris dataset and save it as a `.pkl`.


In [13]:
# 1) Train
X, y = load_iris(return_X_y=True)
clf = RandomForestClassifier(n_estimators=10, random_state=0).fit(X, y)

# 2) Write artifacts at the ROOT (no subfolder). Must be named model.pkl
ARTIFACT_ROOT = pathlib.Path("sklearn_artifacts")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

with open(ARTIFACT_ROOT / "model.pkl", "wb") as f:
    pickle.dump(clf, f)

# (optional debug)
with open(ARTIFACT_ROOT / "sklearn_version.txt", "w") as f:
    f.write(sklearn.__version__)

# Quick local sanity-check: should not raise
with open(ARTIFACT_ROOT / "model.pkl", "rb") as f:
    _ = pickle.load(f)



## 3) Initialize Vertex AI
Initialize the SDK with your project, region, and a staging bucket.


In [14]:

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET)
print("Initialized Vertex AI")


Initialized Vertex AI



## 4) Upload model to Vertex AI Model Registry
We use the prebuilt **scikit-learn** prediction container so you don't have to write any serving code.


In [16]:

model = aiplatform.Model.upload(
    display_name=DISPLAY_NAME,
    artifact_uri=str(ARTIFACT_ROOT),  # SDK uploads this folder to GCS
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-5:latest",
)
model.wait()

print("Model uploaded. Resource name:", model.resource_name)



## 5) Create an Endpoint and Deploy the model


In [17]:

endpoint = aiplatform.Endpoint.create(display_name=f"iris-endpoint-{TIMESTAMP}")
endpoint.wait()

print("Endpoint created:", endpoint.resource_name)


Endpoint created: projects/104115398803/locations/us-east4/endpoints/8763648633095585792


In [18]:
model.deploy(
    endpoint=endpoint,
    machine_type="n1-standard-2",
    traffic_split={"0": 100},
)
print("Deployed to:", endpoint.resource_name)


Deployed to: projects/104115398803/locations/us-east4/endpoints/8763648633095585792



## 6) Online prediction
Send a couple of instances to the endpoint. We reuse `X` from the Iris dataset.


In [21]:

instances = [X[0].tolist(), X[1].tolist()]
prediction = endpoint.predict(instances=instances)
print("Predictions:", prediction.predictions)


Predictions: [0.0, 0.0]



## 7) Batch prediction
Batch prediction reads from **GCS** and writes outputs back to **GCS**.

### 7.1 Create a small JSONL input set and upload to GCS
Each line in JSONL is one instance: `{"instances": [<features>]}` for scikit-learn prebuilt container.


In [22]:

# Create local batch input directory
os.makedirs(BATCH_INPUT_DIR, exist_ok=True)

# Write a tiny JSONL file
batch_file = os.path.join(BATCH_INPUT_DIR, "inputs.jsonl")
with open(batch_file, "w") as f:
    for i in range(5):
        payload = {"instances": X[i].tolist()}
        f.write(json.dumps(payload) + "\n")
print("Wrote batch inputs to:", batch_file)

# Upload to GCS
client = storage.Client(project=PROJECT_ID)
bucket_name = BUCKET.replace("gs://", "").split("/")[0]
prefix = "/".join(BUCKET.replace("gs://", "").split("/")[1:])
gcs_dir = f"batch_inputs/{TIMESTAMP}/"
if prefix:
    gcs_dir = f"{prefix.rstrip('/')}/{gcs_dir}"

bucket = client.bucket(bucket_name)
blob = bucket.blob(f"{gcs_dir}inputs.jsonl")
blob.upload_from_filename(batch_file)
print(f"Uploaded to gs://{bucket_name}/{gcs_dir}inputs.jsonl")

print("GCS input prefix:", f"gs://{bucket_name}/{gcs_dir}")
print("GCS output prefix:", GCS_OUTPUT_PREFIX)


Wrote batch inputs to: batch_inputs_20251009-021406/inputs.jsonl
Uploaded to gs://instr-cs795-fall25-hqin-1-arasm002/batch_inputs/20251009-021406/inputs.jsonl
GCS input prefix: gs://instr-cs795-fall25-hqin-1-arasm002/batch_inputs/20251009-021406/
GCS output prefix: gs://instr-cs795-fall25-hqin-1-arasm002/batch_outputs/20251009-021406/



### 7.2 Launch batch prediction job


In [ ]:
bp_job = model.batch_predict(
    job_display_name=f"iris-batch-{TIMESTAMP}",
    gcs_source=[f"gs://{bucket_name}/{gcs_dir}*.jsonl"],  # make sure this glob matches files
    gcs_destination_prefix=GCS_OUTPUT_PREFIX,
    instances_format="jsonl",
    predictions_format="jsonl",
    machine_type="n1-standard-2",     # <-- required for custom models
    starting_replica_count=1,         # optional
    max_replica_count=2,              # optional
)
print("Batch prediction job started:", bp_job.resource_name)
bp_job.wait()
print("Batch prediction job completed.")



## 9) (Optional) Cleanup
Undeploy the model and delete the endpoint to stop charges when you're done.


In [ ]:

# Uncomment to cleanup when finished:
# endpoint.undeploy_all()
# endpoint.delete()
# print("Endpoint undeployed and deleted.")


# 10) Project Report

This report summarizes the process of training a simple machine learning model, deploying it to Google Cloud's Vertex AI, and using the deployed model for predictions.

**Objective:**
The primary goal of this exercise was to demonstrate a basic end-to-end MLOps workflow using Vertex AI. This involved:
1. Training a scikit-learn model locally.
2. Uploading the model to the Vertex AI Model Registry.
3. Deploying the model to a live endpoint for online predictions.
4. Executing both online (real-time) and batch prediction jobs.

**Process Summary:**
A `RandomForestClassifier` was trained on the Iris dataset and saved locally. The model artifact was then uploaded to the Vertex AI Model Registry, using a pre-built serving container. An endpoint was created, and the model was deployed to it, making it available to serve prediction requests.